In [4]:
from pathlib import Path
from urllib.parse import quote

import subprocess
import pandas as pd

repo = '/kaggle/working/matmul_cuda' if Path('/kaggle/working').is_dir() else '/content/matmul_cuda'
repo_url = 'https://github.com/Sushil2006/matmul_cuda.git'

if Path('/kaggle/working').is_dir():
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GH_TOKEN')
else:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
assert token, 'Add GH_TOKEN to the notebook secrets.'
auth_url = f'https://x-access-token:{quote(token, safe="")}@github.com/Sushil2006/matmul_cuda.git'

if Path(repo, '.git').is_dir():
    !git -C {repo} remote set-url origin {auth_url}
    !git -C {repo} pull --ff-only
else:
    !git clone {auth_url} {repo}

!git -C {repo} remote set-url origin {repo_url}

%cd {repo}

!nvcc -std=c++20 -O3 benchmark.cu kernels/naive.cu kernels/smem_tiled.cu kernels/block_tiled.cu kernels/smem_bank_conflict_free.cu -lcublas -o matmul

Cloning into '/kaggle/working/matmul_cuda'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 61 (delta 33), reused 49 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 343.40 KiB | 2.35 MiB/s, done.
Resolving deltas: 100% (33/33), done.
/kaggle/working/matmul_cuda
nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [7]:
import torch

p = torch.cuda.get_device_properties(0)

print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))
print("VRAM (GiB):", round(p.total_memory / 2**30, 2))
print("SM count:", p.multi_processor_count)
print("Max threads / block:", p.max_threads_per_block)
print("Shared memory / block (KiB):", p.shared_memory_per_block // 1024)
print("Shared memory / SM (KiB):", p.shared_memory_per_multiprocessor // 1024)
print("Threads / block:", p.max_threads_per_block)
print("Threads / SM:", p.max_threads_per_multi_processor)

GPU: Tesla T4
Compute capability: (7, 5)
VRAM (GiB): 14.56
SM count: 40
Max threads / block: 1024
Shared memory / block (KiB): 48
Shared memory / SM (KiB): 64
Threads / block: 1024
Threads / SM: 1024


# Shared-Memory Tiled FP32 GEMM

## Experiment: tile-configuration sweep

- Goal: measure how `BM`, `BN`, and `BK` affect a shared-memory GEMM where one thread computes one output element.
- Matrix shape: `M=N=K=8192`; all values are FP32.
- Timing: CUDA events, one warm-up launch, then the average of three timed launches.
- Correctness: each result is checked against cuBLAS FP32 output.


In [8]:
sizes = [8192]
configs = [
    (16, 16, 8),
    (16, 16, 16),
    (16, 16, 32),
    (32, 8, 16),
    (8, 32, 16),
    (32, 32, 8),
    (32, 32, 16),
    (32, 32, 32),
    (32, 32, 64),
    (32, 32, 128),
]
rows = []

for size in sizes:
    for bm, bn, bk in configs:
        print(f'running N={size} BM={bm} BN={bn} BK={bk}', flush=True)
        completed = subprocess.run(
            ['./matmul', '--kernel', 'smem', '--M', str(size), '--N', str(size), '--K', str(size),
             '--BM', str(bm), '--BN', str(bn), '--BK', str(bk)],
            capture_output=True, text=True, check=True)
        metrics = dict(item.split('=', 1) for item in completed.stdout.strip().splitlines()[-1].split())
        print(f'  time_ms={metrics["time_ms"]}', flush=True)
        rows.append({
            'N': size, 'BM': bm, 'BN': bn, 'BK': bk,
            'time_ms': float(metrics['time_ms']),
            'tflops': float(metrics['tflops']),
            'max_abs_error': float(metrics['max_abs_error']),
            'status': metrics['status'],
        })

df = pd.DataFrame(rows).sort_values(['N', 'time_ms']).reset_index(drop=True)
df

running N=8192 BM=16 BN=16 BK=8


KeyboardInterrupt: 

## Results and interpretation

- The best tested configuration was `(BM, BN, BK) = (32, 32, 128)`: `1205.35 ms` and `0.912 TFLOP/s`.
- For `32 x 32` tiles, increasing `BK` from `8` to `128` reduced runtime from `2111.46 ms` to `1205.35 ms` (`1.75x` faster).
- For fixed `BM` and `BN`, larger `BK` does not change global-memory reuse. It reduces tile-loop overhead and synchronization frequency.

```text
tile iterations = K / BK
barriers = 2 * K / BK

BK=8   -> 1024 tile iterations, 2048 barriers
BK=128 ->   64 tile iterations,  128 barriers
```

- Ignoring the final C store, input arithmetic intensity per tile is approximately:

```text
BM * BN / (2 * (BM + BN)) FLOP/byte

16 x 16 -> 4 FLOP/byte
32 x 32 -> 8 FLOP/byte
32 x 8  -> 3.2 FLOP/byte
```

- A `32 x 32` block has `1024` threads, equal to this T4's `1024 threads/SM` limit, so it uses one resident block per SM while filling the thread capacity.
- `(32, 32, 128)` uses `32 KiB` shared memory per block: `(32 + 32) * 128 * 4`; this fits the `48 KiB` per-block limit.


## Thread-tiled configuration sweep

- This kernel keeps shared-memory tiles but lets each thread compute a `TM x TN` output window in registers.
- The configs below vary `BK`, thread-tile shape, and output-tile shape without a full Cartesian-product sweep.


In [9]:
block_configs = [
    (64, 64, 8, 4, 4),
    (64, 64, 16, 4, 4),
    (64, 64, 32, 4, 4),
    (64, 64, 8, 8, 4),
    (64, 64, 8, 4, 8),
    (128, 64, 8, 8, 4),
    (64, 128, 8, 4, 8),
    (64, 64, 8, 1, 16),
    (64, 64, 8, 16, 1),
    (64, 64, 64, 4, 4),
    (128, 128, 32, 8, 8),
]
block_rows = []

for size in sizes:
    for bm, bn, bk, tm, tn in block_configs:
        print(f'running N={size} BM={bm} BN={bn} BK={bk} TM={tm} TN={tn}', flush=True)
        completed = subprocess.run(
            ['./matmul', '--kernel', 'block', '--M', str(size), '--N', str(size), '--K', str(size),
             '--BM', str(bm), '--BN', str(bn), '--BK', str(bk), '--TM', str(tm), '--TN', str(tn)],
            capture_output=True, text=True, check=True)
        metrics = dict(item.split('=', 1) for item in completed.stdout.strip().splitlines()[-1].split())
        print(f'  time_ms={metrics["time_ms"]}', flush=True)
        block_rows.append({
            'N': size, 'BM': bm, 'BN': bn, 'BK': bk, 'TM': tm, 'TN': tn,
            'time_ms': float(metrics['time_ms']),
            'tflops': float(metrics['tflops']),
            'max_abs_error': float(metrics['max_abs_error']),
            'status': metrics['status'],
        })

block_df = pd.DataFrame(block_rows).sort_values(['N', 'time_ms']).reset_index(drop=True)
block_df

running N=8192 BM=64 BN=64 BK=8 TM=4 TN=4


KeyboardInterrupt: 

## Thread-tiled results and interpretation

- The best tested configuration was `(128, 64, 8, 8, 4)`: `409.743 ms` and `2.683 TFLOP/s`. It is about `2.94x` faster than the best one-output-per-thread shared-memory kernel.
- Increasing the output tile from `64 x 64` to `128 x 64` improved the comparable `8 x 4` thread-tile case from `470.515 ms` to `409.743 ms` (`1.15x` faster), consistent with greater global-memory reuse.
- `(128, 128, 32, 8, 8)` was close behind at `416.458 ms`. Its larger output tile increases reuse, while its `8 x 8` thread tile maintains 64 accumulators per thread and uses `32 KiB` of shared memory per block.
- For the `64 x 64`, `4 x 4` thread-tile case, `BK=16` was best. `BK=32` was effectively the same, while `BK=64` regressed: the synchronization savings did not compensate for the larger shared-memory footprint.
- For `BK=8`, `8 x 4` gives `THREADS_X=16`, so one warp spans two tiled y rows; `4 x 8` gives `THREADS_X=8`, so it spans four.
- The A-tile row spacing between those y groups is `8 * 8 = 64` floats for `8 x 4` and `4 * 8 = 32` floats for `4 x 8`. Both map to the same bank modulo 32, producing 2-way versus 4-way A-tile bank conflicts; this makes `8 x 4` faster (`470.515 ms` versus `575.944 ms`).
- The 1D cases were slower: `16 x 1` took `659.928 ms`, while `1 x 16` took `1283.380 ms`. With `TN=16`, a warp spans many output rows because `blockDim.x = BN / TN = 4`, creating unfavorable shared-memory B-read bank behavior. `TN=1` keeps more warp lanes on consecutive columns, so it performs better, but both lose to 2D tiling.

### Theoretical occupancy: `64 x 64`, `4 x 4`

- Each block has `256` threads (`8` warps). The T4 supports `1024` threads (`32` warps), `64 KiB` shared memory, and 16 blocks per SM.
- For `BK=16`, `ptxas` allocates `75` registers/thread and `8 KiB` shared memory/block. Registers limit residency to three blocks: `3 * 256 = 768` active threads, or `24 / 32 = 75%` theoretical occupancy.
- Active blocks/SM = `min(block limit, thread limit, warp limit, shared-memory limit, register limit)`.

| BK | Shared memory/block | Shared-memory block limit | Register block limit | Active blocks/SM | Theoretical occupancy |
|---:|---:|---:|---:|---:|---:|
| 8 | 4 KiB | 16 | 3 | 3 | 75% |
| 16 | 8 KiB | 8 | 3 | 3 | 75% |
| 32 | 16 KiB | 4 | 3 | 3 | 75% |
| 64 | 32 KiB | 2 | 3 | 2 | 50% |

- At `BK=64`, shared memory becomes the binding resource: active warps fall from `24` to `16`, reducing latency hiding and causing the runtime regression.


In [22]:
finalists = {'A': (128, 64, 8, 8, 4), 'B': (128, 128, 32, 8, 8)}
orders = [('A', 'B'), ('B', 'A'), ('B', 'A'), ('A', 'B')]
finalist_rows = []

for batch, order in enumerate(orders, start=1):
    for config in order:
        bm, bn, bk, tm, tn = finalists[config]
        print(f'batch {batch}/{len(orders)}: {config} BM={bm} BN={bn} BK={bk} TM={tm} TN={tn}', flush=True)
        completed = subprocess.run(
            ['./matmul', '--kernel', 'block', '--M', '16384', '--N', '16384', '--K', '16384',
             '--BM', str(bm), '--BN', str(bn), '--BK', str(bk), '--TM', str(tm), '--TN', str(tn),
             '--runs', '1', '--no-verify'],
            capture_output=True, text=True, check=True)
        metrics = dict(item.split('=', 1) for item in completed.stdout.strip().splitlines()[-1].split())
        print(f'  time_ms={metrics["time_ms"]}', flush=True)
        finalist_rows.append({
            'batch': batch, 'config': config, 'BM': bm, 'BN': bn, 'BK': bk, 'TM': tm, 'TN': tn,
            'time_ms': float(metrics['time_ms']), 'tflops': float(metrics['tflops']),
        })

finalists_df = pd.DataFrame(finalist_rows)
paired = finalists_df.pivot(index='batch', columns='config', values='time_ms')
paired['A_minus_B_ms'] = paired['A'] - paired['B']
median_difference = paired['A_minus_B_ms'].median()
winner_id = 'A' if median_difference < 0 else 'B'
winner = finalists[winner_id]
print(f'FINAL WINNER: {winner_id} = (BM, BN, BK, TM, TN) = {winner}')
print(f'  median(A - B) = {median_difference:.3f} ms')
paired

batch 1/4: A BM=128 BN=64 BK=8 TM=8 TN=4
  time_ms=3315.88
batch 1/4: B BM=128 BN=128 BK=32 TM=8 TN=8
  time_ms=3284.33
batch 2/4: B BM=128 BN=128 BK=32 TM=8 TN=8
  time_ms=3274.58
batch 2/4: A BM=128 BN=64 BK=8 TM=8 TN=4
  time_ms=3529.44
batch 3/4: B BM=128 BN=128 BK=32 TM=8 TN=8
  time_ms=3493.13
batch 3/4: A BM=128 BN=64 BK=8 TM=8 TN=4
  time_ms=3759
batch 4/4: A BM=128 BN=64 BK=8 TM=8 TN=4
  time_ms=3841.84
batch 4/4: B BM=128 BN=128 BK=32 TM=8 TN=8
  time_ms=3594.76
FINAL WINNER: B = (BM, BN, BK, TM, TN) = (128, 128, 32, 8, 8)
  median(A - B) = 250.970 ms


config,A,B,A_minus_B_ms
batch,,,
1,3315.88,3284.33,31.55
2,3529.44,3274.58,254.86
3,3759.00,3493.13,265.87
4,3841.84,3594.76,247.08


## Final configuration

- On `M=N=K=16384`, `(128, 128, 32, 8, 8)` won by a median `250.970 ms` (`6.9%`) over `(128, 64, 8, 8, 4)`.
- The `128 x 128` output tile increases global-memory reuse, and `BK=32` reduces K-tile phases from `2048` to `512`.


In [25]:
size = 8192
bank_conflict_rows = []

for bm, bn, bk, tm, tn in block_configs:
    print(f'running N={size} BM={bm} BN={bn} BK={bk} TM={tm} TN={tn}', flush=True)
    completed = subprocess.run(
        ['./matmul', '--kernel', 'bank-free', '--M', str(size), '--N', str(size), '--K', str(size),
         '--BM', str(bm), '--BN', str(bn), '--BK', str(bk), '--TM', str(tm), '--TN', str(tn)],
        capture_output=True, text=True, check=True)
    metrics = dict(item.split('=', 1) for item in completed.stdout.strip().splitlines()[-1].split())
    print(f'  time_ms={metrics["time_ms"]}', flush=True)
    bank_conflict_rows.append({
        'N': size, 'BM': bm, 'BN': bn, 'BK': bk, 'TM': tm, 'TN': tn,
        'time_ms': float(metrics['time_ms']),
        'tflops': float(metrics['tflops']),
        'max_abs_error': float(metrics['max_abs_error']),
        'status': metrics['status'],
    })

bank_conflict_df = pd.DataFrame(bank_conflict_rows).sort_values(['N', 'time_ms']).reset_index(drop=True)
bank_conflict_df

running N=8192 BM=64 BN=64 BK=8 TM=4 TN=4
  time_ms=561.61
running N=8192 BM=64 BN=64 BK=16 TM=4 TN=4
  time_ms=519.471
running N=8192 BM=64 BN=64 BK=32 TM=4 TN=4
  time_ms=513.926
running N=8192 BM=64 BN=64 BK=8 TM=8 TN=4
  time_ms=494.33
running N=8192 BM=64 BN=64 BK=8 TM=4 TN=8
  time_ms=485.696
running N=8192 BM=128 BN=64 BK=8 TM=8 TN=4
  time_ms=441.361
running N=8192 BM=64 BN=128 BK=8 TM=4 TN=8
  time_ms=447.694
running N=8192 BM=64 BN=64 BK=8 TM=1 TN=16
  time_ms=854.118
running N=8192 BM=64 BN=64 BK=8 TM=16 TN=1
  time_ms=853.614
running N=8192 BM=64 BN=64 BK=64 TM=4 TN=4
  time_ms=559.935
running N=8192 BM=128 BN=128 BK=32 TM=8 TN=8
  time_ms=305.593


,N,BM,BN,BK,TM,TN,time_ms,tflops,max_abs_error,status
0,8192,128,128,32,8,8,305.593,3.59796,0.0,PASS
1,8192,128,64,8,8,4,441.361,2.49118,0.0,PASS
2,8192,64,128,8,4,8,447.694,2.45594,0.0,PASS
3,8192,64,64,8,4,8,485.696,2.26378,0.0,PASS
4,8192,64,64,8,8,4,494.330,2.22425,0.0,PASS
5,8192,64,64,32,4,4,513.926,2.13944,0.0,PASS
6,8192,64,64,16,4,4,519.471,2.11660,0.0,PASS
7,8192,64,64,64,4,4,559.935,1.96364,0.0,PASS
8,8192,64,64,8,4,4,561.610,1.95778,0.0,PASS
9,8192,64,64,8,16,1,853.614,1.28807,0.0,PASS


## Bank-conflict-free summary

- All tested `8192 x 8192` configurations passed correctness verification.
- `(128, 128, 32, 8, 8)` was the fastest at `305.593 ms` (`3.598 TFLOP/s`): `26.6%` faster than the same block-tiled configuration and `25.4%` faster than the previous overall best.
  - Previously, its `TM=8`, `TN=8` thread tile made warp lanes access distinct A rows and B columns that repeatedly mapped to the same shared-memory banks. The new kernel interleaves row/column ownership and XOR-swizzles A's shared-memory K index, spreading those accesses across banks while preserving the logical matrix values.
  - For A specifically, `blockDim.x=16` makes one warp span `threadIdx.y=0` and `1`. The old reads `As[tm][k]` and `As[8 + tm][k]` are `8 * BK = 256` float words apart; with `BK=32`, both map to the same bank (`256 % 32 = 0`). The new layout selects consecutive logical rows and stores their `k` values at different XOR-swizzled columns, so the two broadcasts use different banks.
- The largest gain was `(64, 64, 8, 1, 16)`, improving from `1283.380 ms` to `854.118 ms` (`33.4%`). Its original `TN=16` layout had unfavorable shared-memory B-tile accesses.
- `(64, 64, 8, 16, 1)` regressed because its original accesses were already bank-conflict-free; swizzling added inner-loop address calculations without removing stalls.


In [26]:
size = 8192
bm, bn, bk, tm, tn = 64, 64, 8, 16, 1
orders = [('block', 'bank-free'), ('bank-free', 'block')] * 3
regression_rows = []

# Alternate launch order to reduce clock and thermal bias.
for batch, order in enumerate(orders, start=1):
    for kernel in order:
        print(f'batch {batch}/{len(orders)} kernel={kernel}', flush=True)
        completed = subprocess.run(
            ['./matmul', '--kernel', kernel, '--M', str(size), '--N', str(size), '--K', str(size),
             '--BM', str(bm), '--BN', str(bn), '--BK', str(bk), '--TM', str(tm), '--TN', str(tn),
             '--runs', '3', '--no-verify'],
            capture_output=True, text=True, check=True)
        metrics = dict(item.split('=', 1) for item in completed.stdout.strip().splitlines()[-1].split())
        print(f'  time_ms={metrics["time_ms"]}', flush=True)
        regression_rows.append({'batch': batch, 'kernel': kernel, 'time_ms': float(metrics['time_ms'])})

regression_df = pd.DataFrame(regression_rows)
regression_summary = regression_df.groupby('kernel')['time_ms'].agg(['median', 'mean', 'min', 'max'])
regression_summary['vs_block_pct'] = 100 * (regression_summary['median'] / regression_summary.loc['block', 'median'] - 1)
regression_summary

batch 1/6 kernel=block
batch 1/6 kernel=bank-free
batch 2/6 kernel=bank-free
batch 2/6 kernel=block
batch 3/6 kernel=block
batch 3/6 kernel=bank-free
batch 4/6 kernel=bank-free
batch 4/6 kernel=block
batch 5/6 kernel=block
batch 5/6 kernel=bank-free
batch 6/6 kernel=bank-free
batch 6/6 kernel=block


,median,mean,min,max,vs_block_pct
kernel,,,,,
bank-free,843.3310,843.426333,830.075,857.931,34.780173
block,625.7085,625.289000,611.709,637.728,0.000000


### Why `(64, 64, 8, 16, 1)` regresses

- The original inner loop is already bank-conflict-free: `TN=1` makes B reads consecutive, and every warp has one `threadIdx.y`, so A reads broadcast.
- In the original PTX, the 16 A loads use one base address plus constant offsets. The swizzled PTX must compute separate XOR/add addresses for those loads in every K iteration.
- `ptxas` reports the same 4 KiB shared memory and fewer registers for bank-free (`64` vs `81`), so this is not an occupancy loss; the extra address dependency has no conflicts to repay it.

In [5]:
# One output block preserves the winning configuration's warp and shared-memory access pattern.
!ncu --target-processes all --set basic --launch-skip 1 --launch-count 1 ./matmul --kernel block --M 128 --N 128 --K 128 --BM 128 --BN 128 --BK 32 --TM 8 --TN 8 --runs 1 --no-verify

Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'
==PROF== Connected to process 298 (/kaggle/working/matmul_cuda/matmul)
==ERROR== ERR_NVGPUCTRPERM - The user does not have permission to access NVIDIA GPU Performance Counters on the target device 0. For instructions on enabling permissions and to get more information see https://developer.nvidia.com/ERR_NVGPUCTRPERM
kernel=block M=128 N=128 K=128 BM=128 BN=128 BK=32 TM=8 TN=8 runs=1 time_ms=0.539136 tflops=0.00777968 max_abs_error=0 status=SKIPPED
==PROF== Disconnected from process 298


In [ ]:
# Same launch shape as above; only the bank-conflict-free kernel changes.
!ncu --target-processes all --set basic --launch-skip 1 --launch-count 1 ./matmul --kernel bank-free --M 128 --N 128 --K 128 --BM 128 --BN 128 --BK 32 --TM 8 --TN 8 --runs 1 --no-verify

## Vectorized global-load sweep

This repeats the bank-conflict-free sweep with the same matrix size, configurations, verification, and default timing runs. Re-run the compilation cell above first so `matmul` includes `smem_vectorized.cu`.


In [10]:
size = 8192
vectorized_rows = []

for bm, bn, bk, tm, tn in block_configs:
    print(f'running N={size} BM={bm} BN={bn} BK={bk} TM={tm} TN={tn}', flush=True)
    completed = subprocess.run(
        ['./matmul', '--kernel', 'vectorized', '--M', str(size), '--N', str(size), '--K', str(size),
         '--BM', str(bm), '--BN', str(bn), '--BK', str(bk), '--TM', str(tm), '--TN', str(tn)],
        capture_output=True, text=True, check=True)
    metrics = dict(item.split('=', 1) for item in completed.stdout.strip().splitlines()[-1].split())
    print(f'  time_ms={metrics["time_ms"]}', flush=True)
    vectorized_rows.append({
        'N': size, 'BM': bm, 'BN': bn, 'BK': bk, 'TM': tm, 'TN': tn,
        'time_ms': float(metrics['time_ms']),
        'tflops': float(metrics['tflops']),
        'max_abs_error': float(metrics['max_abs_error']),
        'status': metrics['status'],
    })

vectorized_df = pd.DataFrame(vectorized_rows).sort_values(['N', 'time_ms']).reset_index(drop=True)
comparison_df = bank_conflict_df.merge(
    vectorized_df, on=['N', 'BM', 'BN', 'BK', 'TM', 'TN'], suffixes=('_bank_free', '_vectorized'))
comparison_df['vectorized_speedup_pct'] = 100 * (comparison_df['time_ms_bank_free'] / comparison_df['time_ms_vectorized'] - 1)
comparison_df.sort_values('time_ms_vectorized').reset_index(drop=True)

running N=8192 BM=64 BN=64 BK=8 TM=4 TN=4


CalledProcessError: Command '['./matmul', '--kernel', 'vectorized', '--M', '8192', '--N', '8192', '--K', '8192', '--BM', '64', '--BN', '64', '--BK', '8', '--TM', '4', '--TN', '4']' returned non-zero exit status 1.